# extra 現代資料庫速覽

**資料庫管理**・統計系三年級・自學補充　<a href="https://colab.research.google.com/github/chang-ye-tu/db/blob/master/notebooks/extra_modern.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

課程首頁：[github.com/chang-ye-tu/db](https://github.com/chang-ye-tu/db)・大綱：[syllabus.md](https://github.com/chang-ye-tu/db/blob/master/syllabus.md)・專題：[projects.md](https://github.com/chang-ye-tu/db/blob/master/projects.md)

課後甜點：**U06–U09 內幕支線的自學延伸**——列式、LSM、鍵值／文件／圖、向量檢索、Text-to-SQL。每一節獨立可跳讀；所有玩具都可跑。

> **投影片式 notebook 使用法**：上課跟著往下走，程式格按 `Shift+Enter` 執行；左側「目錄」可跳節。回家可以重跑、改參數做實驗——**講義是可以跑的**。
>
> 開始前建議：檔案 → 在雲端硬碟中儲存副本，改動才會留下來。

## 0. 這本怎麼用

- **時機**：專題交完的寒假、或報告週的等待空檔。不考、不交，純粹讓你「懂的地圖」再大一圈。
- **路線**：每節獨立。§1–§2 接 U08（分析引擎）、§3 接 U06–U07（儲存與索引）、§4–§6 是三種新資料模型、§7 接全學期的 AI 協作。
- **精神不變**：每個「新型資料庫」都是**對某種工作負載的特化**——看懂它「為誰放棄了什麼」，就看懂它。

| 節 | 主題 | 接續 |
|---|---|---|
| §1 | OLTP vs OLAP：存法決定快法 | U08 對決的原理版 |
| §2 | DuckDB 深遊（SUMMARIZE／PIVOT／QUALIFY） | U03、U08 |
| §3 | **LSM-tree 玩具實作**（寫入猛獸的秘密） | U06 pager、U07 B+ tree |
| §4 | 鍵值（Redis）／文件（Mongo）／圖——各 10 分鐘 | U04 建模、U03 遞迴 |
| §5 | SQLite JSON 進階：json_each 與 JSON 欄位索引 | U09 |
| §6 | 向量檢索與 RAG：從 cosine 到語意搜尋 | U09 |
| §7 | Text-to-SQL 與 AI × DB | 全學期【AI 協作】 |

# §1 OLTP vs OLAP：存法決定快法

同一批資料、兩種擺法：

```
 列式（row store）：一筆訂單的所有欄位貼在一起
   [oid|city|amount] [oid|city|amount] [oid|city|amount] …
   → 「撈整筆」一次到位（App 顯示訂單詳情 = OLTP）

 欄式（column store）：同一欄的所有值貼在一起
   [oid oid oid …] [city city city …] [amount amount amount …]
   → 「掃整欄」一次到位（SUM(amount) = OLAP），還超好壓縮
```

用 Python 把兩種擺法都做出來，量給你看：

In [ ]:
import time
import numpy as np

N = 300_000
rng = np.random.default_rng(42)
amounts = rng.integers(10, 2000, N)
cities = rng.choice(["台中", "台北", "高雄"], N)

# 擺法一：列式（list of dict——每列一包）
row_store = [{"oid": i, "city": c, "amount": int(a)} for i, (c, a) in enumerate(zip(cities, amounts))]
# 擺法二：欄式（每欄一條 numpy 陣列）
col_store = {"oid": np.arange(N), "city": cities, "amount": amounts}

t = time.time()
s1 = sum(r["amount"] for r in row_store)                # 掃整欄，但每列都要拆包
t_row = time.time() - t
t = time.time()
s2 = int(col_store["amount"].sum())                     # 整條陣列一次算（向量化）
t_col = time.time() - t
assert s1 == s2
print(f"SUM(amount)：列式 {t_row*1000:6.1f} ms ｜ 欄式 {t_col*1000:6.2f} ms → 快 {t_row/t_col:,.0f} 倍")

t = time.time()
r1 = row_store[123_456]                                 # 撈「一整筆」
t_pt_row = time.time() - t
t = time.time()
r2 = {k: col_store[k][123_456] for k in col_store}      # 欄式要跑三條陣列各撈一格
t_pt_col = time.time() - t
print(f"撈一整筆　 ：列式 {t_pt_row*1e6:6.1f} µs ｜ 欄式 {t_pt_col*1e6:6.1f} µs → 列式勝")
print()
print("→ U08 的 SQLite vs DuckDB 對決，勝負一半在這（另一半是向量化執行）。")
print("  沒有「比較好的擺法」，只有「對到工作負載的擺法」——這句話是整本補充講義的主旋律。")

In [ ]:
# 欄式的隱藏紅利：同欄放一起「超好壓縮」——run-length（RLE）＋字典編碼的手算帳
import itertools
sorted_cities = np.sort(cities)                       # 欄式引擎常按欄排序後儲存
runs = len([1 for _, _g in itertools.groupby(sorted_cities)])
raw_bytes = sum(len(c.encode()) for c in cities)      # 天真存法：每列存整個字串
dict_bytes = N * 1 + sum(len(c.encode()) for c in set(cities))   # 字典編碼：每列 1 byte 代碼
rle_bytes = runs * (1 + 4) + sum(len(c.encode()) for c in set(cities))  # 排序後 RLE：每段 (代碼, 長度)
print(f"city 欄 {N:,} 列：原始 {raw_bytes/1e6:.1f} MB → 字典編碼 {dict_bytes/1e6:.2f} MB"
      f" → 排序＋RLE {rle_bytes} bytes（{runs} 段）")
print("→ 只有 3 種值的欄，30 萬列可以壓到幾十 bytes——掃描時連解壓都不用（直接對代碼算）。")
print("  Parquet／DuckDB 的檔案小、掃得快，一半的功勞在這種欄式壓縮。")

# §2 DuckDB 深遊：分析師的瑞士刀

U03 用過它直查 DataFrame／CSV／Parquet、U08 看過它贏下聚合對決。再學三招日常神器：

In [ ]:
import sys, importlib.util, subprocess
if importlib.util.find_spec("duckdb") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
import duckdb
import pandas as pd

odf = pd.DataFrame({
    "oid": np.arange(50_000),
    "city": rng.choice(["台中", "台北", "高雄", "台南"], 50_000),
    "cat": rng.choice(["飲料", "零食", "文具"], 50_000, p=[.5, .3, .2]),
    "amount": np.round(rng.lognormal(4.2, .6, 50_000)).astype(int),
})
print("道具：5 萬列訂單 DataFrame odf（duckdb 直接把同名變數當表查）")
print(duckdb.sql("SELECT COUNT(*), ROUND(AVG(amount),1) FROM odf").df().to_string(index=False))

In [ ]:
# 招式一 SUMMARIZE：一句話拿到整表的敘述統計（describe 的 SQL 版，還附 NULL 數與近似 NDV）
smry = duckdb.sql("SUMMARIZE odf").df()
show_cols = [c for c in ["column_name", "column_type", "min", "max", "approx_unique", "null_percentage"]
             if c in smry.columns]
print(smry[show_cols].to_string(index=False))
print("\n→ 拿到陌生資料的第一句話就是它——比逐欄手查快十倍。")

In [ ]:
# 招式二 PIVOT：交叉表不用再手寫 SUM(CASE …)（U03 的樞紐，原生語法版）
print(duckdb.sql("""
    PIVOT odf ON cat USING SUM(amount) GROUP BY city ORDER BY city
""").df().to_string(index=False))
print("\n→ SQLite 要寫三個 SUM(CASE WHEN…)；DuckDB 一句 PIVOT。欄位動態長出來。")

In [ ]:
# 招式三 QUALIFY：window 函數的「HAVING」——每組 top-k 不用再包一層 CTE
print(duckdb.sql("""
    SELECT city, cat, SUM(amount) AS rev,
           ROW_NUMBER() OVER (PARTITION BY city ORDER BY SUM(amount) DESC) AS rn
    FROM odf
    GROUP BY city, cat
    QUALIFY rn <= 2                 -- 直接過濾 window 結果！
    ORDER BY city, rn
""").df().to_string(index=False))
print("\n→ U03 的「CTE + ROW_NUMBER + WHERE rn<=k」三層，在 DuckDB 壓成一層。")
print("  語法糖不改變觀念——你在 U03 學的心智模型全部通用。")

In [ ]:
# 小練習工作區：用 QUALIFY 一句話寫出「各分類金額最高的那一筆訂單（oid）」
# 提示：PARTITION BY cat ORDER BY amount DESC；驗收：3 列（每分類一列）
# TODO
# print(duckdb.sql("""...""").df())




# §3 LSM-tree 玩具實作：寫入猛獸的秘密

U07 的 B+ tree 讀得快，但**每筆寫入都要找到樹上的位置**（隨機 I/O）。
如果系統是「狂寫少讀」（日誌、感測器、聊天訊息）呢？LSM-tree（log-structured merge tree）反過來想：

```
 寫入 ──► memtable（記憶體，排序中）──滿了──► 整批刷成一個「已排序段」（循序寫！）
                                              segment_3（最新）
 讀 k ──► 先查 memtable ──沒有──► 從新到舊     segment_2      各段內二分搜
                                  一段段找     segment_1（最舊）
 背景 compaction ──► 把多個段合併成一大段（丟掉舊版本）──► 讀的段數變少
```

- **寫**：永遠 append／整批刷——把隨機寫變循序寫（U06 說過循序快得多）。
- **代價**：讀要翻好幾段（read amplification）＋背景要一直合併（write amplification）。
- RocksDB／LevelDB／Cassandra 的心臟就是它。60 行做一顆：

In [ ]:
import bisect

class MiniLSM:
    def __init__(self, cap=1000):
        self.memtable = {}                      # 最新資料（dict＝隨手寫，之後排序）
        self.segments = []                      # list of (keys_sorted, values)——舊 → 新
        self.cap = cap
        self.stats = {"flush": 0, "compact": 0}

    def put(self, k, v):
        self.memtable[k] = v                    # O(1)！寫入只碰記憶體
        if len(self.memtable) >= self.cap:
            self._flush()

    def _flush(self):                           # 滿了：整批排序、循序刷出去成一段
        keys = sorted(self.memtable)
        self.segments.append((keys, [self.memtable[k] for k in keys]))
        self.memtable = {}
        self.stats["flush"] += 1

    def get(self, k):
        if k in self.memtable:                  # 先看最新
            return self.memtable[k]
        for keys, vals in reversed(self.segments):      # 從新段往舊段找
            i = bisect.bisect_left(keys, k)
            if i < len(keys) and keys[i] == k:
                return vals[i]
        return None

    def compact(self):                          # 把所有段合併成一段（新蓋舊）
        merged = {}
        for keys, vals in self.segments:        # 舊 → 新，後者覆蓋前者
            merged.update(zip(keys, vals))
        keys = sorted(merged)
        self.segments = [(keys, [merged[k] for k in keys])]
        self.stats["compact"] += 1

db = MiniLSM(cap=1000)
print("MiniLSM 就緒——put 只碰記憶體、flush 循序刷段、get 從新到舊翻段、compact 合併")

In [ ]:
# 正確性驗證：一萬筆寫入（含大量覆寫），跟 dict 標準答案對照
truth = {}
rng2 = np.random.default_rng(7)
for k, v in zip(rng2.integers(0, 3000, 10_000).tolist(), range(10_000)):
    db.put(k, v); truth[k] = v

print(f"寫入一萬筆後：memtable {len(db.memtable)} 筆＋磁碟段 {len(db.segments)} 段"
      f"（flush 了 {db.stats['flush']} 次）")
sample_keys = rng2.integers(0, 3500, 500).tolist()
assert all(db.get(k) == truth.get(k) for k in sample_keys)
print("✅ 500 筆抽查（含不存在的 key）全對——覆寫語意正確：新段蓋舊段")

db.compact()
print(f"compaction 後：{len(db.segments)} 段、共 {len(db.segments[0][0]):,} 個唯一 key（舊版本被丟掉）")
assert all(db.get(k) == truth.get(k) for k in sample_keys)
print("✅ 合併後答案不變、讀取只需翻一段——這就是背景 compaction 換來的讀取效率")

In [ ]:
# 為什麼說「寫入猛獸」？跟 U07 的 sorted list（B-tree 的直覺替身）比寫入速度
batch = rng2.integers(0, 10**9, 30_000).tolist()

t = time.time()
lsm = MiniLSM(cap=1000)
for i, k in enumerate(batch):
    lsm.put(k, i)
t_lsm = time.time() - t

t = time.time()
sorted_arr = []
for k in batch:
    bisect.insort(sorted_arr, k)                # 每筆都要挪陣列（B-tree 世界的「就地維序」）
t_insort = time.time() - t

print(f"寫 3 萬筆：MiniLSM {t_lsm*1000:6.0f} ms ｜ 就地維序 {t_insort*1000:6.0f} ms"
      f" → LSM 快 {t_insort/t_lsm:.0f} 倍")
print()
print("→ 代價在讀：LSM 可能翻好幾段（真品用 bloom filter 幫每段先「聞」一下有沒有 key）。")
print("  一句話總結：B+ tree 為讀最佳化、LSM 為寫最佳化——又是「工作負載決定引擎」。")

In [ ]:
# 加碼：那個「聞一下」的 bloom filter，20 行做給你看——用兩個 hash 把 key 壓進一排 bit
class MiniBloom:
    def __init__(self, m=8192):
        self.m, self.bits = m, bytearray(m // 8)

    def _hashes(self, k):
        h1 = hash(("salt1", k)) % self.m
        h2 = hash(("salt2", k)) % self.m
        return h1, h2

    def add(self, k):
        for h in self._hashes(k):
            self.bits[h // 8] |= (1 << (h % 8))

    def might_have(self, k):                    # False＝絕對沒有；True＝「可能有」（要真的去查）
        return all(self.bits[h // 8] & (1 << (h % 8)) for h in self._hashes(k))

bloom = MiniBloom()
seg_keys = set(rng2.integers(0, 10**9, 5000).tolist())      # 想像這是某個 segment 的 key 集
for k in seg_keys:
    bloom.add(k)

miss_probe = [int(k) for k in rng2.integers(0, 10**9, 10_000)]   # 幾乎都不在段裡的查詢
skipped = sum(1 for k in miss_probe if not bloom.might_have(k))
fp = sum(1 for k in miss_probe if bloom.might_have(k) and k not in seg_keys)
assert all(bloom.might_have(k) for k in list(seg_keys)[:100])    # 在裡面的永遠說「可能有」
print(f"一萬次「不在段裡」的查詢：{skipped:,} 次被 bloom 直接擋掉（免翻段），誤報 {fp} 次")
print(f"記憶體成本：{bloom.m // 8:,} bytes 服務 {len(seg_keys):,} 個 key")
print("→ 「絕不漏報、偶爾誤報」的機率濾網——LSM 每段配一個，miss 查詢幾乎不再碰磁碟。")
print("  統計系加映：誤報率 ≈ (1 - e^(-kn/m))^k——參數怎麼調，是道很漂亮的機率題。")

In [ ]:
# LSM 練習工作區：幫 MiniLSM 加上 delete(k)——提示：LSM 不能「就地刪」，要寫「墓碑（tombstone）」
# 規格：delete(k) = put(k, TOMBSTONE 哨兵值)；get() 讀到墓碑回 None；compact() 時把墓碑真正丟掉
# TODO：
# TOMBSTONE = object()
# def delete(self, k): ...
# （自測：put→delete→get 是 None；compact 後 key 從段裡消失）


print("工作區就緒——寫完你就懂了：LSM 世界連「刪除」都是一筆追加寫入。")

# §4 鍵值／文件／圖：三種資料模型，各 10 分鐘

## 4.1 鍵值（Redis 型）：只剩 get/put，換來極致速度

Redis ＝ 一個活在記憶體、帶過期時間（TTL）的大 dict，用途：快取、session、排行榜、限流。玩具版：

In [ ]:
class MiniRedis:
    def __init__(self):
        self.data = {}                                  # k -> (value, 過期時間戳或 None)
        self.clock = 0.0                                # 玩具用假時鐘（可重現）

    def set(self, k, v, ttl=None):
        self.data[k] = (v, self.clock + ttl if ttl else None)

    def get(self, k):
        if k not in self.data:
            return None
        v, expiry = self.data[k]
        if expiry is not None and self.clock >= expiry:  # 過期＝當作不存在（惰性刪除）
            del self.data[k]
            return None
        return v

r = MiniRedis()
r.set("hot:report:2026-08", "（昨晚算好的報表 JSON）", ttl=60)      # 快取 60 秒
r.set("session:abc123", "user=佳蓉", ttl=30)
print("剛存完：", r.get("hot:report:2026-08"), "|", r.get("session:abc123"))
r.clock += 45                                           # 快轉 45 秒
print("45 秒後：", r.get("hot:report:2026-08"), "|", r.get("session:abc123"), "← session 先過期")
assert r.get("session:abc123") is None and r.get("hot:report:2026-08") is not None
print()
print("→ 用途對號：專題裡「每晚重算的報表快取」（U04 反正規化那招）就是這個模式的 SQL 版。")
print("  Redis 放棄了查詢語言與跨鍵交易，換到微秒級延遲與每秒百萬操作。")

## 4.2 文件（MongoDB 型）：整包 JSON 當一筆

U09 玩過 `json_extract`。文件庫把「payload 是 JSON」推到底：沒有固定 schema、巢狀隨你放。
**代價**：資料庫幫不了你把關（U02 的約束都沒了）、join 要自己來。
SQLite 的 JSON 函數讓你**兩邊的好處各拿一半**——固定欄位好好開、雜項進 JSON（§5 繼續深入）。

## 4.3 圖（Neo4j 型）：關係本身是主角

「誰跟誰是朋友的朋友？」——關聯表照樣存得下（edges 表），遞迴 CTE（U03！）照樣查得動：

In [ ]:
import sqlite3
g = sqlite3.connect(":memory:")
g.executescript("""
CREATE TABLE knows(a TEXT, b TEXT);          -- 邊：a 認識 b（無向就存兩筆或查雙向）
INSERT INTO knows VALUES
 ('佳蓉','威廷'), ('佳蓉','雅筑'), ('威廷','孟軒'), ('雅筑','芷瑄'),
 ('孟軒','子涵'), ('芷瑄','宇翔'), ('威廷','雅筑');
""")
rows = g.execute("""
    WITH RECURSIVE reach(person, dist) AS (
        SELECT '佳蓉', 0
        UNION
        SELECT CASE WHEN e.a = rc.person THEN e.b ELSE e.a END, rc.dist + 1
        FROM knows e JOIN reach AS rc ON rc.person IN (e.a, e.b)
        WHERE rc.dist < 2                       -- 只走兩步：朋友的朋友
    )
    SELECT person, MIN(dist) AS min_dist FROM reach GROUP BY person ORDER BY min_dist, person""").fetchall()
for person, d in rows:
    print(f"  距離 {d}：{person}")
d = dict(rows)
assert d["孟軒"] == 2 and d["芷瑄"] == 2 and "子涵" not in d      # 子涵在三步外——兩步圈看不到他
print()
print("→ 子涵、宇翔在三步之外，兩步圈自然沒有他們——把 WHERE rc.dist < 2 調大就擴圈。")
print("  SQL 做得到「幾步內」的遍歷；圖資料庫贏在「不定深度＋上億邊」的效能與查詢語法（Cypher）。")
print("  你的分類樹、組織階層用 SQL 遞迴綽綽有餘——別為小圖搬大砲。")

### 隨堂快答：三個系統各交給誰？

① 即時聊天訊息流（每秒上萬則、幾乎只 append）　② 電商商品目錄（欄位固定＋每商品一些雜七雜八規格）
③ 「你可能認識的人」推薦（三度人脈、上億邊）

<details><summary>答案</summary>
① LSM 系（Cassandra/RocksDB）——狂寫少讀的天菜；② 關聯為主＋JSON 欄放雜項規格（§5 的混合式；純文件庫也常見，但訂單交易端仍要關聯）；③ 圖資料庫——不定深度遍歷是它的主場。
判準永遠是那句：**工作負載長什麼樣、你願意放棄什麼。**
</details>

# §5 SQLite JSON 進階：json_each 展開 ＋ 給 JSON 欄位建索引

兩招讓「半結構」資料在關聯世界過好日子：

In [ ]:
# 招式一 json_each：把 JSON 陣列「展開成列」——瞬間回到你熟悉的關聯世界
j = sqlite3.connect(":memory:")
j.execute("CREATE TABLE orders_j(id INTEGER PRIMARY KEY, doc TEXT)")
j.executemany("INSERT INTO orders_j(doc) VALUES (?)", [
    ('{"who": "佳蓉", "items": [{"n": "珍奶", "q": 2}, {"n": "蛋餅", "q": 1}]}',),
    ('{"who": "威廷", "items": [{"n": "珍奶", "q": 1}]}',),
    ('{"who": "孟軒", "items": [{"n": "咖啡", "q": 3}, {"n": "珍奶", "q": 1}]}',),
])
rows = j.execute("""
    SELECT json_extract(o.doc, '$.who')      AS 誰,
           json_extract(item.value, '$.n')   AS 品項,
           json_extract(item.value, '$.q')   AS 數量
    FROM orders_j o, json_each(o.doc, '$.items') AS item      -- 一列 JSON → N 列明細！
""").fetchall()
for r_ in rows:
    print("  ", r_)
print("\n珍奶總杯數：", j.execute("""
    SELECT SUM(json_extract(item.value, '$.q'))
    FROM orders_j o, json_each(o.doc, '$.items') AS item
    WHERE json_extract(item.value, '$.n') = '珍奶'""").fetchone()[0])
print("→ 展開之後 GROUP BY／join／window 全部照舊——「文件」與「關聯」不是二選一。")

In [ ]:
# 招式二：JSON 欄位也能吃索引——generated column（U02）＋ index（U07）的合體技
j.execute("ALTER TABLE orders_j ADD COLUMN who TEXT GENERATED ALWAYS AS (json_extract(doc, '$.who'))")
j.execute("CREATE INDEX idx_who ON orders_j(who)")
plan = "; ".join(r_[3] for r_ in j.execute(
    "EXPLAIN QUERY PLAN SELECT * FROM orders_j WHERE who = '佳蓉'"))
print("查詢計畫：", plan)
assert "idx_who" in plan
print("→ JSON 裡常查的鍵「抽出來」變 generated column、建索引——彈性與效能兼得。")
print("  這招在正式的 PostgreSQL（JSONB＋expression index）一模一樣通用。")

# §6 向量檢索與 RAG：從 cosine 到語意搜尋

U09 的玩具版把句子變**詞頻向量**；真實世界用 embedding 模型把語意壓進幾百維。流程一樣：

```
 建庫：文件 ──embedding──► 向量 ──► 存進向量索引（HNSW/IVF：近似最近鄰）
 查詢：問題 ──embedding──► 向量 ──► 找最近的 k 篇 ──► 塞給 LLM 當上下文 ＝ RAG
```

這裡把玩具升級成「可用的檢索器」：n-gram 向量（不用外部模型也能抓到部分相似）＋ top-k API：

In [ ]:
corpus = [
    "SQLite 的交易保證原子性與持久性", "B+ tree 讓百萬列點查只要三次讀頁",
    "window function 可以算移動平均與排名", "圖書館系統的借閱與預約流程設計",
    "遞迴 CTE 能展開階層與產生日曆", "lost update 要用原子更新來防",
    "合成資料要有長尾與季節節律", "EXPLAIN 查詢計畫會顯示掃描或走索引",
    "向量資料庫用最近鄰搜尋做語意檢索", "Gradio 讓資料庫應用長出網頁介面",
]

def ngrams(s, n=2):
    """字元 bigram（中文友善：不用斷詞）"""
    return [s[i:i+n] for i in range(len(s) - n + 1)]

from collections import Counter
vocab = sorted({g for s in corpus for g in ngrams(s)})
gram_idx = {g: i for i, g in enumerate(vocab)}

def vectorize(s):
    v = np.zeros(len(vocab))
    for g, c in Counter(ngrams(s)).items():
        if g in gram_idx:
            v[gram_idx[g]] = c
    norm = np.linalg.norm(v)
    return v / norm if norm else v

mat = np.array([vectorize(s) for s in corpus])

def search(query, k=3):
    scores = mat @ vectorize(query)
    return [(float(scores[i]), corpus[i]) for i in np.argsort(-scores)[:k]]

for q in ["怎麼防止更新不見", "查詢很慢想看計畫", "幫我做語意搜尋"]:
    print(f"🔍 {q}")
    for score, text in search(q):
        print(f"   {score:.3f}  {text}")
    print()
assert search("怎麼防止更新不見")[0][1].startswith("lost update")
assert search("查詢很慢想看計畫")[0][1].startswith("EXPLAIN")
print("✅ 沒有任何 AI 模型，bigram + cosine 已經能「大概懂你」——")
print("   換上真 embedding（語意向量），錯字、同義詞、跨語言都能對上；那就是 RAG 的檢索層。")

In [ ]:
# 混合檢索（實務標配）：先用「結構化條件」過濾、再用向量排序——SQL 與向量各司其職
meta = ["內幕", "內幕", "SQL", "應用", "SQL", "內幕", "應用", "內幕", "現代", "應用"]   # 每篇的分類標籤

def hybrid_search(query, category, k=3):
    idxs = [i for i, m in enumerate(meta) if m == category]      # ① WHERE category = ?（結構化過濾）
    if not idxs:
        return []
    scores = mat[idxs] @ vectorize(query)                        # ② 剩下的才算向量相似度
    order = np.argsort(-scores)[:k]
    return [(float(scores[o]), corpus[idxs[o]]) for o in order]

print("🔍「排名跟平均」限定分類 = SQL：")
for score, text in hybrid_search("排名跟平均", "SQL"):
    print(f"   {score:.3f}  {text}")
assert hybrid_search("排名跟平均", "SQL")[0][1].startswith("window")
print()
print("→ 「金額 > 1000」永遠是 WHERE 的事、「跟這段最像」才是向量的事——")
print("  真品（pgvector／sqlite-vec）就是把這兩步放進同一句 SQL 裡。")

### 通往真品的三步

1. **embedding**：`sentence-transformers` 或各家 API——一句話變 384～3072 維向量；
2. **向量索引**：資料小（<10 萬）直接 numpy 內積掃描就夠（今天這招）；大了再上 FAISS／`sqlite-vec`／pgvector 的 HNSW **近似**最近鄰（犧牲一點 recall 換千倍速度——又是取捨！）；
3. **RAG**：檢索到的 top-k 文段＋問題一起給 LLM——「開書考」永遠比「憑記憶」可靠。

一句提醒：**向量檢索不取代 SQL**——混合檢索（filter + vector，上一格親手做過）是實務標配。

# §7 Text-to-SQL 與 AI × DB

這學期每個單元的【AI 協作】其實都在做同一件事：**把自然語言需求變成可驗證的資料庫操作**。
Text-to-SQL 就是它的自動化版。先玩一個「上古時代」的規則式玩具，體會這件事哪裡難：

In [ ]:
import re as re2

def rule_text2sql(q):
    """規則式 Text-to-SQL（1970s 風味）：只認「各<欄>的<總|平均|筆數><欄>」句型"""
    m = re2.match(r"各(\w+)的(總|平均|筆數)(\w*)", q)
    if not m:
        return None
    grp_col, agg, val_col = m.groups()
    fn = {"總": f"SUM({val_col})", "平均": f"ROUND(AVG({val_col}), 1)", "筆數": "COUNT(*)"}[agg]
    return f"SELECT {grp_col}, {fn} FROM odf GROUP BY {grp_col}"

for q in ["各city的總amount", "各cat的平均amount", "各city的筆數", "上個月台中賣最好的品類是什麼"]:
    sql = rule_text2sql(q)
    print(f"「{q}」")
    if sql:
        print(f"   → {sql}")
        print(duckdb.sql(sql).df().to_string(index=False).replace("\n", "\n     "))
    else:
        print("   → 🤷 句型不認得（規則式的天花板：世界上的問法無限多）")
    print()

規則式在第四句就陣亡——這正是 LLM 接手的地方：它「什麼問法都接得住」。
但接得住 ≠ 答得對。**LLM Text-to-SQL 的四大翻車**與你早就會的解法：

| 翻車 | 長相 | 你的解法（哪個單元學的） |
|---|---|---|
| 幻覺欄位 | `WHERE customer_name = …`（表裡叫 cname） | prompt 附 schema（U03 模板） |
| 粒度錯 | join 後 COUNT 變人次 | 粒度盤問（U03 陷阱 1.6） |
| 分母錯 | 及格率把 NULL 當 0 | 分母三問（U02 及格率陷阱） |
| 跑得動的錯 | 語法全對、語意全錯 | **雙引擎交叉驗證**（U03 黃金驗證法） |

**AI 時代資料庫課的結論**：SQL 不會因為 AI 會寫就不用學——恰恰相反，
「看得懂、驗得出、改得動」的人，才是能放心讓 AI 寫第一版的人。這學期你練的就是這個。

## 課程總複習地圖（兼告別）

| 你現在會的 | 在哪學的 | 這本補充把它延伸到 |
|---|---|---|
| SQL 全套＋window | U02–U03 | DuckDB 的 SUMMARIZE/PIVOT/QUALIFY |
| 建模與正規化 | U04 | 文件模型的「彈性 vs 把關」取捨 |
| 交易與競態防護 | U05–U06、U09 | Redis 放棄交易換速度的理由 |
| page／B+ tree | U06–U07 | LSM：為寫最佳化的另一條路（＋bloom filter） |
| 查詢引擎與列式 | U08 | 欄式擺法的第一性原理 |
| 遞迴 CTE | U03 | 圖遍歷 |
| AI 協作與驗收 | 每個單元 | Text-to-SQL 與 RAG（混合檢索） |

進階書單（都值得，照興趣挑）：Petrov《Database Internals》（引擎內幕）、
Kleppmann《Designing Data-Intensive Applications》（分散式與資料系統的聖經）、
Winand《SQL Performance Explained》（索引實務最好的一本）。

**期末快樂。資料庫這門手藝，畢業後每一年都會再感謝自己一次。**